In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
from natsort import natsorted
from tqdm.notebook import tqdm
from os.path import join as pjoin
import plotly.graph_objects as go
from statsmodels.formula.api import ols
from scipy.stats import pearsonr, spearmanr, zscore
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/snake_plots'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 0.2 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians
chunks = 4 ## number of time bins for cell-cell correlations

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

### Example snake plot for a single session.

In [ ]:
## Settings
mouse = 'mc51'
experiment = 'MultiCon_Imaging5'
day_of_int = 16
correct_dir = True
only_running = True 
normalize_plot = True
smooth_data = False
colorscale = 'gray_r'

fig = pf.custom_graph_template(x_title='Position (cm)', y_title='Neuron', height=500, width=400)

exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, f'{mouse}_{data_type}_{day_of_int}.nc'))[data_type]
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## use cells that only meet activity threshold
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['water'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['water'])].values)

if np.isnan(reward_one_pos):
    reward_one_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == sdata.attrs['reward_one']].values)

if np.isnan(reward_two_pos):
    reward_two_pos = np.mean(sdata['lin_position'][sdata['lick_port'] == sdata.attrs['reward_two']].values)

if smooth_data:
    smoothed_ar = ctn.moving_average_xarray(sdata, ksize=8)
else:
    smoothed_ar = sdata.copy()
neural_data, position_data = ctn.subset_correct_dir_and_running(smoothed_ar, correct_dir=correct_dir, only_running=only_running, velocity_thresh=velocity_thresh)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size)
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
resh_pop = tuning_curves.reshape((tuning_curves.shape[1], tuning_curves.shape[0]))
centers_resh = np.argmax(resh_pop, axis=1)
order_resh = np.argsort(centers_resh)

if normalize_plot:
    max_vals = np.max(resh_pop, axis=1)
    for uid in np.arange(0, resh_pop.shape[0]):
        resh_pop[uid, :] = resh_pop[uid, :] / max_vals[uid]

fig.add_trace(go.Heatmap(x=bins * conversion, y=np.arange(0, sdata.shape[0]), z=resh_pop[order_resh], coloraxis='coloraxis1'))
# for pos in [reward_one_pos, reward_two_pos]:
#     fig.add_vline(x=(pos - reward_bin_size) * conversion, line_width=1, line_color='red', opacity=1, line_dash='dash')
#     fig.add_vline(x=(pos + reward_bin_size) * conversion, line_width=1, line_color='red', opacity=1, line_dash='dash')
for pos in [reward_one_pos, reward_two_pos]:
    fig.add_vline(x=pos * conversion, line_width=1, opacity=0.7, line_color='red', line_dash='dash')
fig.update_layout(
    title={
        'text': f"{sdata.attrs['group']}",
        'x': 0.5,               
        'xanchor': 'center',    
        'font': {
            'family': "Arial",
            'size': 26
        }
    }
)
fig.update_layout(coloraxis1={'colorscale': colorscale})
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_snake_plot.png'), width=350, height=500)

In [ ]:
## Plot heatmap of cell firing on the circle track
neuron = 229
x0 = 90
x1 = 540
y0 = -10
y1 = 460
amount = 40

fig = pf.custom_graph_template(x_title='', y_title='', width=550)
H, x_edges, y_edges = np.histogram2d(neural_data['x'].values, neural_data['y'].values, bins=40, weights=neural_data[neuron].values)
hnorm = H / np.max(H)
hnorm[hnorm == 0] = np.nan
fig.add_trace(go.Heatmap(x=x_edges, y=y_edges, z=hnorm, colorscale='reds'))

## Create outer and inner circles
fig.add_shape(type="circle",
    xref="x", yref="y",
    x0=x0, y0=y0, x1=x1, y1=y1,
    line=dict(color='black', width=4), layer='below',
    fillcolor='rgba(0,0,0,0)'
)
fig.add_shape(type="circle",
    xref="x", yref="y",
    x0=x0 + amount, y0=y0 + amount, x1=x1 - amount, y1=y1 - amount,
    line=dict(color='black', width=4), layer='below',
    fillcolor='rgba(0,0,0,0)'
)

for port in np.arange(1, 9):
    x = np.mean(sdata['x'][(sdata['lick_port'] == port) & (sdata['correct_dir'])].values)
    y = np.mean(sdata['y'][(sdata['lick_port'] == port) & (sdata['correct_dir'])].values)

    if port == 1:
        x = x + 35
    elif port == 2:
        x = x - 8
    elif port == 3:
        x = x - 15
        y = y - 15
    elif port == 4:
        x = x - 8
        y = y - 8
    elif port == 5:
        y = y - 4
    elif port == 6:
        x = x + 8
        y = y - 8
    elif port == 7:
        x = x + 20
    elif port == 8:
        x = x + 20
        y = y - 10

    if port in [sdata.attrs['reward_one'], sdata.attrs['reward_two']]:
        color = 'red'
    else:
        color = 'black'
    fig.add_trace(go.Scattergl(x=[x], y=[y], showlegend=False, marker_color=color, marker_size=7, name=f'{port}',
                               marker=dict(line=dict(width=1.5, color='black'))))

fig.add_annotation(x=np.mean([np.max(x_edges), np.min(x_edges)]) - 100, y=np.mean([np.max(y_edges), np.min(y_edges)]) - 100, 
                   text=f'SI: {np.round(neural_data['skaggs_info'].values[neuron], 2)}', showarrow=False, font_size=16)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_neuron{neuron}_rate_map.svg'), width=550, height=500)

In [ ]:
np.mean(sdata['x'][(sdata['lick_port'] == 1) & (sdata['correct_dir'])].values)

### All 20 days snake plots for a single mouse. Cells are not cross-registered across sessions.

In [ ]:
mouse = 'mc58'
experiment = 'MultiCon_Imaging6'
correct_dir = True 
only_running = True
normalize_plot = True
colorscale = 'cividis'

if mouse in control_mice:
    title_list = control_list 
else:
    title_list = session_list

fig = pf.custom_graph_template(x_title='', y_title='', rows=4, columns=5, width=1100, height=1100,
                               titles=title_list, font_size=15)

exp_path = pjoin(dpath, f'{experiment}/output/aligned_minian/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')
for idx, session in enumerate(natsorted(os.listdir(mpath))):
    if '21' in session:
        pass 
    else:
        if (mouse == 'mc42') & (idx > 14):
            idx += 1
        elif (mouse == 'mc43') & (idx > 11):
            idx += 1
        elif (mouse == 'mc44') & (idx > 7):
            idx += 1
        elif (mouse == 'mc46') & (idx > 9):
            idx += 1
        elif (mouse == 'mc52') & (idx > 2):
            idx += 1
        elif (mouse == 'mc55') & (idx > 2):
            idx += 1
        
        if idx < 5:
            row, col = 1, idx + 1
        elif (idx >= 5) & (idx < 10):
            row, col = 2, idx - 4
        elif (idx >= 10) & (idx < 15):
            row, col = 3, idx - 9
        else:
            row, col = 4, idx - 14
        
        sdata = xr.open_dataset(pjoin(mpath, f'{session}'))[data_type]
        reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['water'])].values)
        reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['water'])].values)

        neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, velocity_thresh=velocity_thresh)
        population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size)
        tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
        resh_pop = tuning_curves.reshape((tuning_curves.shape[1], tuning_curves.shape[0]))
        centers_resh = np.argmax(resh_pop, axis=1)
        order_resh = np.argsort(centers_resh)

        if normalize_plot:
            max_vals = np.max(resh_pop, axis=1)
            for uid in np.arange(0, resh_pop.shape[0]):
                resh_pop[uid, :] = resh_pop[uid, :] / max_vals[uid]

        fig.add_trace(go.Heatmap(x=bins, y=np.arange(0, sdata.shape[0]), z=resh_pop[order_resh], coloraxis='coloraxis1'), row=row, col=col)
        for pos in [reward_one_pos, reward_two_pos]:
            fig.add_vline(x=pos, line_width=1, line_color='red', opacity=1, row=row, col=col)

fig.update_yaxes(autorange='reversed')
fig.update_yaxes(title='Neuron ID', col=1)
fig.update_xaxes(title='Lin Position (rad)', row=4)
fig.update_layout(coloraxis1={'colorscale': colorscale})
fig.show()
fig.write_image(pjoin(fig_path, f'snake_plots_all_sessions_{mouse}.png'), width=1100, height=1100)